In [1]:
import pandas as pd

# Dataset SAIH geolocalizado
saih = pd.read_parquet("../data/processed/saih_geolocalizado_2020_2024.parquet")
print("=== SAIH geolocalizado ===")
print(f"Filas: {len(saih):,}")
print(f"Columnas: {list(saih.columns)}")
print()
print(saih.head(3))
print()
print(f"Rango de fechas: {saih['fecha'].min()} a {saih['fecha'].max()}")
print(f"Estaciones únicas: {saih['descripcion'].nunique()}")

=== SAIH geolocalizado ===
Filas: 31,697
Columnas: ['fecha', 'subcuenca', 'descripcion', 'provincia', 'precipitacion_mm', 'match_final', 'lat', 'lon', 'fuente_coords']

       fecha              subcuenca  \
0 2020-01-19  CENIA-BAJO MAESTRAZGO   
1 2020-01-19  CENIA-BAJO MAESTRAZGO   
2 2020-01-19  CENIA-BAJO MAESTRAZGO   

                                       descripcion provincia  \
0          Cervera del Maestre Cervera del Maestre        CS   
1       Embalse de Ulldecona La Pobla de Benifassa        CS   
2  Pluvionivómetro de Fredes La Pobla de Benifassa        CS   

   precipitacion_mm match_final        lat       lon     fuente_coords  
0              30.4         NaN        NaN       NaN         sin_match  
1              15.0   ULLDECONA  40.671433  0.233323  shapefile_miteco  
2              12.6      FREDES  40.695043  0.167687  shapefile_miteco  

Rango de fechas: 2020-01-19 00:00:00 a 2024-11-13 00:00:00
Estaciones únicas: 186


In [2]:
from pathlib import Path

archivos_mdt = list(Path("../data/interim").glob("*.tif"))
print("Ficheros MDT disponibles:")
for f in archivos_mdt:
    print(f"  {f.name}")

Ficheros MDT disponibles:
  mdt_dana_utm30.tif
  mdt_dana_wgs84.tif
  pendiente_dana_utm30.tif


COMIENZA EL CÓDIGO

# Notebook 07 — Ingeniería de variables (feature engineering)

**Objetivo**: a partir del dataset SAIH geolocalizado y el MDT, construir la tabla de variables que alimentará el modelo predictivo.

**Unidad de análisis**: cada fila es una estación-día.

**Grupos de variables**:
1. Lluvia del día
2. Lluvia acumulada (ventanas de 3 y 7 días + humedad antecedente)
3. Topografía (elevación y pendiente, extraídas del MDT)
4. Temporales (mes, temporada de lluvias)
5. Espaciales (subcuenca)

## 1. Carga y preparación del dataset base

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Cargamos el SAIH geolocalizado
saih = pd.read_parquet("../data/processed/saih_geolocalizado_2020_2024.parquet")

# Nos quedamos solo con las estaciones que tienen coordenadas
# (las necesitamos para cruzar con el MDT)
saih = saih[saih["lat"].notna() & saih["lon"].notna()].copy()

# Aseguramos tipo fecha y ordenamos por estación y fecha
saih["fecha"] = pd.to_datetime(saih["fecha"])
saih = saih.sort_values(["descripcion", "fecha"]).reset_index(drop=True)

print(f"Registros con coordenadas: {len(saih):,}")
print(f"Estaciones: {saih['descripcion'].nunique()}")
print(f"Periodo: {saih['fecha'].min().date()} a {saih['fecha'].max().date()}")
saih.head(3)

Registros con coordenadas: 19,747
Estaciones: 115
Periodo: 2020-01-19 a 2024-11-13


,fecha,subcuenca,descripcion,provincia,precipitacion_mm,match_final,lat,lon,fuente_coords
0,2020-01-19,DEL BAJO TURIA,Acequia Moncada El Puig,V,29.4,EL PUIG,39.605677,-0.314405,shapefile_miteco
1,2020-01-20,DEL BAJO TURIA,Acequia Moncada El Puig,V,40.8,EL PUIG,39.605677,-0.314405,shapefile_miteco
2,2020-01-21,DEL BAJO TURIA,Acequia Moncada El Puig,V,18.2,EL PUIG,39.605677,-0.314405,shapefile_miteco


## 2. Lluvia del día y ventanas acumuladas

Calculamos la precipitación acumulada en ventanas móviles. Para cada estación-día sumamos la lluvia de los N días anteriores. Los días sin informe (sin fila) se tratan implícitamente como lluvia 0, porque solo sumamos lo que existe en el dataset dentro de la ventana temporal.

- `precip_dia`: lluvia del propio día
- `precip_3d`: acumulado de los últimos 3 días (incluido el actual)
- `precip_7d`: acumulado de los últimos 7 días (incluido el actual)
- `precip_previa_7d`: lluvia de los 7 días ANTERIORES (humedad antecedente del suelo)

In [4]:
# La lluvia del día es la columna que ya tenemos
saih["precip_dia"] = saih["precipitacion_mm"]


def acumular_ventana(grupo, dias, incluir_actual=True):
    """Para cada fila, suma la lluvia de los 'dias' días previos.
    
    Trabaja por estación. Usa la fecha real, así que si faltan días
    intermedios (sin informe) cuentan como 0 automáticamente.
    """
    grupo = grupo.set_index("fecha").sort_index()
    serie = grupo["precip_dia"]
    resultado = []
    for fecha_actual in serie.index:
        if incluir_actual:
            ini = fecha_actual - pd.Timedelta(days=dias - 1)
            fin = fecha_actual
        else:
            ini = fecha_actual - pd.Timedelta(days=dias)
            fin = fecha_actual - pd.Timedelta(days=1)
        ventana = serie[(serie.index >= ini) & (serie.index <= fin)]
        resultado.append(ventana.sum())
    return pd.Series(resultado, index=serie.index)


# Aplicamos por estación
features_lluvia = []
for estacion, grupo in saih.groupby("descripcion"):
    g = grupo.copy()
    g = g.set_index("fecha").sort_index()
    
    g["precip_3d"] = acumular_ventana(grupo, 3, incluir_actual=True).values
    g["precip_7d"] = acumular_ventana(grupo, 7, incluir_actual=True).values
    g["precip_previa_7d"] = acumular_ventana(grupo, 7, incluir_actual=False).values
    
    g["descripcion"] = estacion
    features_lluvia.append(g.reset_index())

saih = pd.concat(features_lluvia, ignore_index=True)

print("Ventanas de lluvia calculadas.")
print(saih[["descripcion", "fecha", "precip_dia", "precip_3d",
            "precip_7d", "precip_previa_7d"]].head(8))

Ventanas de lluvia calculadas.
               descripcion      fecha  precip_dia  precip_3d  precip_7d  \
0  Acequia Moncada El Puig 2020-01-19        29.4       29.4       29.4   
1  Acequia Moncada El Puig 2020-01-20        40.8       70.2       70.2   
2  Acequia Moncada El Puig 2020-01-21        18.2       88.4       88.4   
3  Acequia Moncada El Puig 2020-01-22        11.0       70.0       99.4   
4  Acequia Moncada El Puig 2020-03-01         0.0        0.0        0.0   
5  Acequia Moncada El Puig 2020-03-13         1.6        1.6        1.6   
6  Acequia Moncada El Puig 2020-03-16        16.0       16.0       17.6   
7  Acequia Moncada El Puig 2020-03-17         6.2       22.2       23.8   

   precip_previa_7d  
0               0.0  
1              29.4  
2              70.2  
3              88.4  
4               0.0  
5               0.0  
6               1.6  
7              17.6  


## 3. Variables topográficas (extraídas del MDT)

Para cada estación, consultamos su elevación y su pendiente en los rásters del MDT. Como una estación es un punto fijo, estos valores son constantes para todas las fechas de esa estación.

Usamos el MDT en proyección WGS84 (lat/lon) para la elevación, y el ráster de pendiente. Si una estación cae fuera del área cubierta por el MDT, su valor quedará nulo y lo trataremos después.

In [5]:
import requests
import time

def consultar_elevacion(puntos, dataset="eudem25m"):
    """Consulta elevación de una lista de (lat, lon) vía OpenTopoData.
    Máximo 100 puntos por petición, 1 petición/segundo."""
    elevaciones = []
    for i in range(0, len(puntos), 100):
        lote = puntos[i:i + 100]
        locations = "|".join(f"{lat},{lon}" for lat, lon in lote)
        url = f"https://api.opentopodata.org/v1/{dataset}"
        r = requests.get(url, params={"locations": locations}, timeout=30)
        r.raise_for_status()
        data = r.json()
        for res in data["results"]:
            elevaciones.append(res["elevation"])
        print(f"  Lote {i//100 + 1}: {len(lote)} puntos consultados")
        time.sleep(1)  # respetar el límite del servicio
    return elevaciones


# Coordenadas únicas por estación
estaciones_coords = (saih[["descripcion", "lat", "lon"]]
                     .drop_duplicates("descripcion")
                     .reset_index(drop=True))

puntos = list(zip(estaciones_coords["lat"], estaciones_coords["lon"]))

print(f"Consultando elevación de {len(puntos)} estaciones...")
estaciones_coords["elevacion"] = consultar_elevacion(puntos)

n_ok = estaciones_coords["elevacion"].notna().sum()
print(f"\nEstaciones con elevación: {n_ok} de {len(estaciones_coords)}")
print(estaciones_coords[["descripcion", "elevacion"]].head(10))

Consultando elevación de 115 estaciones...
  Lote 1: 100 puntos consultados
  Lote 2: 15 puntos consultados

Estaciones con elevación: 115 de 115
                                    descripcion    elevacion
0                       Acequia Moncada El Puig    18.675497
1       Aforo Trasvase Tajo-Segura Los Anguijes   713.216858
2                  Aforo en Cofrentes Cofrentes   346.887604
3                        Aforo en Cuenca Cuenca   920.578796
4  Aforo en Gea de Albarracin Gea de Albarracin  1112.476562
5                Aforo en Huerto Mulet Algemesí    13.933674
6            Aforo en Pajaroncillo Pajaroncillo   943.426575
7                        Aforo en Teruel Teruel   896.138855
8    Aforo en Villalba Alta Perales de Alfambra  1072.492310
9                         Aforo en Zagra Tuejar   570.080200


import rasterio

def extraer_valor_raster(ruta_raster, puntos_latlon, crs_raster_es_latlon=True):
    """Extrae el valor del ráster en cada punto (lat, lon).
    
    puntos_latlon: lista de tuplas (lat, lon)
    Devuelve lista de valores (None si el punto cae fuera).
    """
    valores = []
    with rasterio.open(ruta_raster) as src:
        for lat, lon in puntos_latlon:
            try:
                # rasterio.sample espera (x, y) = (lon, lat)
                muestra = list(src.sample([(lon, lat)]))[0][0]
                # Filtramos valores "nodata"
                if src.nodata is not None and muestra == src.nodata:
                    valores.append(None)
                else:
                    valores.append(float(muestra))
            except Exception:
                valores.append(None)
    return valores


# Coordenadas únicas por estación (no hace falta repetir el cálculo por fecha)
estaciones_coords = (saih[["descripcion", "lat", "lon"]]
                     .drop_duplicates("descripcion")
                     .reset_index(drop=True))

puntos = list(zip(estaciones_coords["lat"], estaciones_coords["lon"]))

# Elevación: del MDT en WGS84
estaciones_coords["elevacion"] = extraer_valor_raster(
    "../data/interim/mdt_dana_wgs84.tif", puntos
)

print("Elevación extraída.")
print(estaciones_coords.head(10))
print(f"\nEstaciones con elevación: {estaciones_coords['elevacion'].notna().sum()}")
print(f"Estaciones sin elevación (fuera del MDT): {estaciones_coords['elevacion'].isna().sum()}")

**Nota importante sobre la cobertura del MDT**: solo ALGUNAS hojas del MDT, así que muchas estaciones SAIH caerán fuera y no tendrán elevación. Esto lo veremos en la salida de la celda anterior.

En la siguiente celda decidimos cómo tratar esos huecos.

In [6]:
import numpy as np

def consultar_pendiente(puntos, dataset="eudem25m", offset_m=250):
    """Calcula la pendiente en cada punto consultando 4 vecinos (N,S,E,O)
    y midiendo el desnivel."""
    # Conversión metros -> grados (aprox. para latitud ~39.5°)
    dlat = offset_m / 111_000
    dlon = offset_m / 85_000

    # Para cada estación generamos 4 vecinos
    todos_puntos = []
    for lat, lon in puntos:
        todos_puntos.append((lat + dlat, lon))  # Norte
        todos_puntos.append((lat - dlat, lon))  # Sur
        todos_puntos.append((lat, lon + dlon))  # Este
        todos_puntos.append((lat, lon - dlon))  # Oeste

    elevaciones_vecinos = consultar_elevacion(todos_puntos, dataset)

    # Calculamos la pendiente de cada estación a partir de sus 4 vecinos
    pendientes = []
    for i in range(0, len(elevaciones_vecinos), 4):
        norte, sur, este, oeste = elevaciones_vecinos[i:i + 4]
        if None in (norte, sur, este, oeste):
            pendientes.append(None)
            continue
        # Gradiente: desnivel / distancia (distancia entre opuestos = 2*offset)
        grad_ns = (norte - sur) / (2 * offset_m)
        grad_eo = (este - oeste) / (2 * offset_m)
        pendiente_grados = np.degrees(np.arctan(np.sqrt(grad_ns**2 + grad_eo**2)))
        pendientes.append(pendiente_grados)
    return pendientes


print(f"Consultando pendiente de {len(puntos)} estaciones (4 vecinos cada una)...")
estaciones_coords["pendiente"] = consultar_pendiente(puntos)

print(f"\nEstaciones con pendiente: {estaciones_coords['pendiente'].notna().sum()}")
print(estaciones_coords[["descripcion", "elevacion", "pendiente"]].head(10))

Consultando pendiente de 115 estaciones (4 vecinos cada una)...
  Lote 1: 100 puntos consultados
  Lote 2: 100 puntos consultados
  Lote 3: 100 puntos consultados
  Lote 4: 100 puntos consultados
  Lote 5: 60 puntos consultados

Estaciones con pendiente: 115
                                    descripcion    elevacion  pendiente
0                       Acequia Moncada El Puig    18.675497   0.600955
1       Aforo Trasvase Tajo-Segura Los Anguijes   713.216858   0.705203
2                  Aforo en Cofrentes Cofrentes   346.887604   5.799316
3                        Aforo en Cuenca Cuenca   920.578796   1.341661
4  Aforo en Gea de Albarracin Gea de Albarracin  1112.476562   3.339677
5                Aforo en Huerto Mulet Algemesí    13.933674   0.309049
6            Aforo en Pajaroncillo Pajaroncillo   943.426575   5.675630
7                        Aforo en Teruel Teruel   896.138855   6.166098
8    Aforo en Villalba Alta Perales de Alfambra  1072.492310  11.312378
9                    

# La pendiente está en el ráster en proyección UTM, no WGS84.
# Para extraerla correctamente necesitamos reproyectar los puntos.
# Como solución simple y robusta, reproyectamos las coordenadas a UTM 30N.

from pyproj import Transformer

transformer = Transformer.from_crs("EPSG:4326", "EPSG:25830", always_xy=True)

puntos_utm = []
for lat, lon in puntos:
    x, y = transformer.transform(lon, lat)
    puntos_utm.append((y, x))  # ojo: la función espera (lat-equiv, lon-equiv)

# Extraemos pendiente con coordenadas UTM
def extraer_valor_raster_utm(ruta_raster, puntos_xy):
    """puntos_xy: lista de tuplas (y, x) en la proyección del ráster."""
    valores = []
    with rasterio.open(ruta_raster) as src:
        for y, x in puntos_xy:
            try:
                muestra = list(src.sample([(x, y)]))[0][0]
                if src.nodata is not None and muestra == src.nodata:
                    valores.append(None)
                else:
                    valores.append(float(muestra))
            except Exception:
                valores.append(None)
    return valores

estaciones_coords["pendiente"] = extraer_valor_raster_utm(
    "../data/interim/pendiente_dana_utm30.tif", puntos_utm
)

print("Pendiente extraída.")
print(estaciones_coords[["descripcion", "elevacion", "pendiente"]].head(10))

## 4. Variables temporales

Extraemos el mes de cada registro y un indicador de temporada de lluvias (septiembre a noviembre, los meses típicos de DANA en el Mediterráneo).

In [7]:
# Unimos las variables topográficas al dataset principal
saih = saih.merge(
    estaciones_coords[["descripcion", "elevacion", "pendiente"]],
    on="descripcion",
    how="left",
)

# Variables temporales
saih["mes"] = saih["fecha"].dt.month
saih["temporada_lluvias"] = saih["mes"].isin([9, 10, 11]).astype(int)

print("Variables temporales añadidas.")
print(saih[["fecha", "mes", "temporada_lluvias"]].drop_duplicates().head(12))

Variables temporales añadidas.
        fecha  mes  temporada_lluvias
0  2020-01-19    1                  0
1  2020-01-20    1                  0
2  2020-01-21    1                  0
3  2020-01-22    1                  0
4  2020-03-01    3                  0
5  2020-03-13    3                  0
6  2020-03-16    3                  0
7  2020-03-17    3                  0
8  2020-03-20    3                  0
9  2020-03-23    3                  0
10 2020-03-25    3                  0
11 2020-03-30    3                  0


## 5. Revisión del dataset de features

Revisamos el dataset completo antes de pasar al etiquetado (notebook 08).

In [8]:
# Seleccionamos las columnas finales del dataset de features
columnas_features = [
    "descripcion", "fecha", "lat", "lon", "subcuenca",
    "precip_dia", "precip_3d", "precip_7d", "precip_previa_7d",
    "elevacion", "pendiente",
    "mes", "temporada_lluvias",
]

dataset = saih[columnas_features].copy()

print(f"Dataset de features: {len(dataset):,} filas, {len(dataset.columns)} columnas")
print()
print("Valores nulos por columna:")
print(dataset.isna().sum())
print()
print("Estadísticas de las variables numéricas:")
print(dataset[["precip_dia", "precip_3d", "precip_7d",
               "precip_previa_7d", "elevacion", "pendiente"]].describe().round(1))

Dataset de features: 19,747 filas, 13 columnas

Valores nulos por columna:
descripcion          0
fecha                0
lat                  0
lon                  0
subcuenca            0
precip_dia           0
precip_3d            0
precip_7d            0
precip_previa_7d     0
elevacion            0
pendiente            0
mes                  0
temporada_lluvias    0
dtype: int64

Estadísticas de las variables numéricas:
       precip_dia  precip_3d  precip_7d  precip_previa_7d  elevacion  \
count     19747.0    19747.0    19747.0           19747.0    19747.0   
mean          8.3       14.5       20.4              13.6      626.8   
std          16.0       28.2       35.6              31.6      429.8   
min           0.0        0.0        0.0               0.0        8.3   
25%           0.2        0.6        1.4               0.0      308.5   
50%           2.8        5.0        8.6               0.8      578.8   
75%          10.0       16.6       24.2              14.0      963.

### CELDA AÑADIDAS PARA LA RECONSTRUCCIÓN DEL MODELO

In [10]:
# === Mejora 2: features de dinámica temporal ===
saih = saih.sort_values(["descripcion", "fecha"]).reset_index(drop=True)

# Tendencia: diferencia entre la lluvia de hoy y la del registro anterior
saih["precip_tendencia"] = saih.groupby("descripcion")["precip_dia"].diff().fillna(0)

# Ratio de lo reciente sobre lo acumulado (evita división por cero)
saih["ratio_reciente"] = saih["precip_dia"] / saih["precip_7d"].replace(0, np.nan)
saih["ratio_reciente"] = saih["ratio_reciente"].fillna(0)

# Pico máximo de los últimos 3 días (incluido hoy)
def maximo_ventana(grupo, dias):
    grupo = grupo.set_index("fecha").sort_index()
    serie = grupo["precip_dia"]
    resultado = []
    for fecha_actual in serie.index:
        ini = fecha_actual - pd.Timedelta(days=dias - 1)
        ventana = serie[(serie.index >= ini) & (serie.index <= fecha_actual)]
        resultado.append(ventana.max())
    return pd.Series(resultado, index=serie.index)

picos = []
for estacion, grupo in saih.groupby("descripcion"):
    g = grupo.set_index("fecha").sort_index()
    g["precip_max_3d"] = maximo_ventana(grupo, 3).values
    g["descripcion"] = estacion
    picos.append(g.reset_index())
saih = pd.concat(picos, ignore_index=True)

# Días consecutivos con lluvia (> 1 mm)
def dias_consecutivos(grupo):
    grupo = grupo.sort_values("fecha")
    llueve = (grupo["precip_dia"] > 1).astype(int)
    contador = []
    c = 0
    for v in llueve:
        c = c + 1 if v == 1 else 0
        contador.append(c)
    return pd.Series(contador, index=grupo.index)

saih["dias_consecutivos_lluvia"] = (
    saih.groupby("descripcion", group_keys=False).apply(dias_consecutivos)
)

print("Nuevas features de dinámica temporal añadidas:")
print(saih[["descripcion", "fecha", "precip_dia", "precip_tendencia",
            "ratio_reciente", "precip_max_3d", "dias_consecutivos_lluvia"]].head(10))

Nuevas features de dinámica temporal añadidas:
               descripcion      fecha  precip_dia  precip_tendencia  \
0  Acequia Moncada El Puig 2020-01-19        29.4               0.0   
1  Acequia Moncada El Puig 2020-01-20        40.8              11.4   
2  Acequia Moncada El Puig 2020-01-21        18.2             -22.6   
3  Acequia Moncada El Puig 2020-01-22        11.0              -7.2   
4  Acequia Moncada El Puig 2020-03-01         0.0             -11.0   
5  Acequia Moncada El Puig 2020-03-13         1.6               1.6   
6  Acequia Moncada El Puig 2020-03-16        16.0              14.4   
7  Acequia Moncada El Puig 2020-03-17         6.2              -9.8   
8  Acequia Moncada El Puig 2020-03-20         9.8               3.6   
9  Acequia Moncada El Puig 2020-03-23        10.4               0.6   

   ratio_reciente  precip_max_3d  dias_consecutivos_lluvia  
0        1.000000           29.4                         1  
1        0.581197           40.8                 

In [11]:
columnas_features_RECONSTRUCCION = [
    "descripcion", "fecha", "lat", "lon", "subcuenca",
    "precip_dia", "precip_3d", "precip_7d", "precip_previa_7d",
    "elevacion", "pendiente",
    "mes", "temporada_lluvias",
    "precip_tendencia", "ratio_reciente", "precip_max_3d",
    "dias_consecutivos_lluvia",
]

dataset_RECONSTRUCCION = saih[columnas_features_RECONSTRUCCION].copy()

print(f"Dataset de features: {len(dataset_RECONSTRUCCION):,} filas, "
      f"{len(dataset_RECONSTRUCCION.columns)} columnas")
print()
print("Valores nulos por columna:")
print(dataset_RECONSTRUCCION.isna().sum())
print()
print("Estadísticas de las variables numéricas:")
print(dataset_RECONSTRUCCION[["precip_dia", "precip_3d", "precip_7d",
      "precip_previa_7d", "elevacion", "pendiente", "precip_tendencia",
      "ratio_reciente", "precip_max_3d", "dias_consecutivos_lluvia"]].describe().round(1))

Dataset de features: 19,747 filas, 17 columnas

Valores nulos por columna:
descripcion                 0
fecha                       0
lat                         0
lon                         0
subcuenca                   0
precip_dia                  0
precip_3d                   0
precip_7d                   0
precip_previa_7d            0
elevacion                   0
pendiente                   0
mes                         0
temporada_lluvias           0
precip_tendencia            0
ratio_reciente              0
precip_max_3d               0
dias_consecutivos_lluvia    0
dtype: int64

Estadísticas de las variables numéricas:
       precip_dia  precip_3d  precip_7d  precip_previa_7d  elevacion  \
count     19747.0    19747.0    19747.0           19747.0    19747.0   
mean          8.3       14.5       20.4              13.6      626.8   
std          16.0       28.2       35.6              31.6      429.8   
min           0.0        0.0        0.0               0.0        8.3   


In [12]:
# OJO: guardamos dataset_RECONSTRUCCION (no 'dataset', que es el viejo)
out_path = Path("../data/processed/features_modelo_RECONSTRUCCION.parquet")
dataset_RECONSTRUCCION.to_parquet(out_path, index=False)
print(f"Guardado: {out_path}")
print(f"  {len(dataset_RECONSTRUCCION):,} filas, "
      f"{len(dataset_RECONSTRUCCION.columns)} columnas")

Guardado: ..\data\processed\features_modelo_RECONSTRUCCION.parquet
  19,747 filas, 17 columnas


In [9]:
# Guardamos el dataset de features (aún sin etiqueta)
out_path = Path("../data/processed/features_modelo.parquet")
dataset.to_parquet(out_path, index=False)
print(f"Guardado: {out_path}")
print(f"  {len(dataset):,} filas")

Guardado: ..\data\processed\features_modelo.parquet
  19,747 filas


## Conclusiones del notebook

Se ha construido el dataset de variables predictoras (features) con una fila por estación-día. Incluye variables de lluvia (actual y acumulada), topográficas (elevación y pendiente) y temporales.

**Pendiente para el siguiente notebook (08)**: definir la etiqueta de riesgo (variable objetivo) mediante la estrategia de umbrales, y entrenar el modelo XGBoost.

"La elevación y pendiente de cada estación se obtuvieron mediante el servicio OpenTopoData sobre el modelo EU-DEM (resolución 25 m), dado que la descarga de la cobertura MDT25 completa de la cuenca del Júcar resultaba inviable. El MDT25 del IGN se reservó para las visualizaciones de relieve a escala de detalle de la zona del evento."